# Model 2 — MobileNetV2

MobileNetV2 uses ImageNet transfer learning with a small classification head. It is included because its depthwise convolutions are efficient for practical CPU inference and it is a strong baseline when the medical dataset is limited.

In [ ]:
from pathlib import Path
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "1")
import json, math, sys
import numpy as np
import tensorflow as tf

# Allow TensorFlow to grow GPU memory as needed instead of reserving a fixed block.
_available_gpus = tf.config.list_physical_devices('GPU')
for _gpu in _available_gpus:
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except RuntimeError:
        pass
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
SERVICE_DIR = next(p for p in candidates if (p / 'app').is_dir() and (p / 'requirements.txt').exists())
sys.path.insert(0, str(SERVICE_DIR))
from app.config import ARTIFACT_DIR, DATASET_CSV, IMAGE_SIZE, SEED
from app.data import load_manifest, split_manifest
from app.labels import CLASS_NAMES
from app.metrics import calculate_classification_metrics
from app.model import build_transfer_model
USE_IMAGENET_WEIGHTS = True  # Change to False when training without internet.
tf.keras.utils.set_random_seed(SEED)
print('Service:', SERVICE_DIR)
print('Dataset:', DATASET_CSV)

In [ ]:
frame = load_manifest()
train, validation, test = split_manifest(frame)
print(f'Usable images: {len(frame):,} | train: {len(train):,} | validation: {len(validation):,} | test: {len(test):,}')
display(frame['label'].value_counts().reindex(CLASS_NAMES).rename('count').to_frame())

In [ ]:
BATCH_SIZE = 8
def make_dataset(dataframe, shuffle=False):
    paths = dataframe['path'].to_numpy()
    labels = dataframe['label_index'].to_numpy(dtype=np.int32)
    def load(path, label):
        image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        return tf.image.resize(image, IMAGE_SIZE), label
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.apply(tf.data.experimental.ignore_errors())
    return dataset.batch(BATCH_SIZE).prefetch(1)
train_dataset = make_dataset(train, True).repeat()
validation_dataset = make_dataset(validation).repeat()
test_dataset = make_dataset(test)
TRAIN_STEPS = math.ceil(len(train) / BATCH_SIZE)
VALIDATION_STEPS = math.ceil(len(validation) / BATCH_SIZE)
weights = compute_class_weight('balanced', classes=np.arange(len(CLASS_NAMES)), y=train['label_index'])
class_weights = {i: float(value) for i, value in enumerate(weights)}
print('Class weights:', class_weights)

In [ ]:
# Recreate the repeated pipelines here so this training cell is safe to rerun.
train_dataset = make_dataset(train, True).repeat()
validation_dataset = make_dataset(validation).repeat()
TRAIN_STEPS = math.ceil(len(train) / BATCH_SIZE)
VALIDATION_STEPS = math.ceil(len(validation) / BATCH_SIZE)
model = build_transfer_model('mobilenetv2', weights='imagenet' if USE_IMAGENET_WEIGHTS else None)
model.summary()
model_path = ARTIFACT_DIR / 'models' / 'mobilenetv2.keras'
model_path.parent.mkdir(parents=True, exist_ok=True)
callbacks = [tf.keras.callbacks.ModelCheckpoint(model_path, monitor='val_accuracy', save_best_only=True), tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True), tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6)]
history = model.fit(train_dataset, validation_data=validation_dataset, epochs=15, steps_per_epoch=TRAIN_STEPS, validation_steps=VALIDATION_STEPS, class_weight=class_weights, callbacks=callbacks, shuffle=False)

In [ ]:
best_model = tf.keras.models.load_model(model_path)
predicted = best_model.predict(test_dataset, verbose=0).argmax(axis=1)
actual = np.concatenate([labels.numpy() for _, labels in test_dataset], axis=0)
metrics = {'model': 'mobilenetv2', **calculate_classification_metrics(actual, predicted)}
print(classification_report(actual, predicted, target_names=CLASS_NAMES, zero_division=0))
(ARTIFACT_DIR / 'models' / 'mobilenetv2_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Saved:', model_path)
print(metrics)